In [ ]:
import os
from pathlib import Path

import pandas as pd

from genpm.modelling.model_utils.cvae_utils import (
    generate_timespan,
    load_artifacts,
    load_training_data,
)
from genpm.modelling.validation import (
    plot_kpi_interactive,
    wide_timespan_to_long,
)

In [ ]:
USER = os.environ.get("USER", "")
SHARED_DIR_PATH = Path(f"/home/{USER}/app/apps/apps/generator/data/shared_dir")
RUN_DIR_PATH = SHARED_DIR_PATH / "artifacts" / "run_4"
WEIGHTS_PATH = RUN_DIR_PATH / "models_weights" / "cvae_lstm_v5_0.weights.h5"
REAL_DATA_PATH = SHARED_DIR_PATH / "preprocessed_dataset" / "final_scaled_only_minmax" / "pm_df_long_indexed_winds"

In [ ]:
CELL_ID = "bts_24/cell_5"   # set this
DATE_START = "2023-12-12"
DATE_END = "2024-03-12"
CHOSEN_KPI = "NR_5096"
SEED = 42

In [ ]:
model, cell_encoder = load_artifacts(
    run_id_path=RUN_DIR_PATH,
    weights_path=WEIGHTS_PATH,
)
training_data = load_training_data(RUN_DIR_PATH)

df_syn = generate_timespan(
    model=model,
    **training_data,
    params_df=None,
    cell_id=CELL_ID,
    date_start=DATE_START,
    date_end=DATE_END,
    seed=SEED,
)
df_syn["distname"] = "synthetic"
long_syn = wide_timespan_to_long(df_syn, keep_window_anchor=False)

In [ ]:
pdf = pd.read_parquet(REAL_DATA_PATH)
pdf["timestamp"] = pd.to_datetime(pdf["window_anchor"]) + pd.to_timedelta(pdf["hour_idx"], unit="h")
long_real = (
    pdf[pdf["distname"] == CELL_ID]
    .drop(columns=["window_anchor", "bts_id", "hour_idx"], errors="ignore")
    .melt(id_vars=["distname", "timestamp"], var_name="kpi_id", value_name="kpi_value")
)
long_real = long_real.assign(distname="real")

In [ ]:
combined = pd.concat([long_syn, long_real], ignore_index=True)
plot_kpi_interactive(combined, kpi_id=CHOSEN_KPI, distname=CELL_ID)